In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from openpyxl import load_workbook
import openpyxl
import os

In [ ]:
# Load all stock price data from Excel file
all_stock_prices = pd.read_excel('Data/all_stock_prices.xlsx')

# Initialize an empty DataFrame to store processed data
all_prices = pd.DataFrame()

In [ ]:
# Exclude the "Date" column from return calculations
columns_to_exclude = ['Date']
columns_to_calculate = [col for col in all_stock_prices.columns if col not in columns_to_exclude]

# Compute logarithmic returns for each column
returns_stock_prices = np.log(
    all_stock_prices[columns_to_calculate] / all_stock_prices[columns_to_calculate].shift(1)
)

In [ ]:
# Add "Date" column to the returns DataFrame
returns_stock_prices['Date'] = all_stock_prices['Date']

# Reorder columns
returns_stock_prices = returns_stock_prices[['Date'] + columns_to_calculate]

# Replace missing values with zero
returns_stock_prices = returns_stock_prices.fillna(0)

# Display the returns DataFrame
print(returns_stock_prices)

In [ ]:
# Output Excel file path
output_file = 'Data/returns_stock_prices.xlsx'

# Save the DataFrame to Excel
returns_stock_prices.to_excel(output_file, index=False)

In [ ]:
# Load date ranges from Excel file
dates = pd.read_excel('Data/shock_periods.xlsx')

# Convert "Date" column to datetime format
returns_stock_prices['Date'] = pd.to_datetime(returns_stock_prices['Date'])

# Initialize a dictionary to store filtered data by date range
filtered_stock_prices_by_date = {}

In [ ]:
# Filter stock returns for each date range and store in dictionary
for index, date_row in dates.iterrows():
    start_date = pd.to_datetime(date_row['start'])
    end_date = pd.to_datetime(date_row['end'])
    
    # Keep only periods with at least 5 days
    if date_row['number of days'] >= 5:
        new_df = returns_stock_prices[
            returns_stock_prices['Date'].between(start_date, end_date)
        ]
        filtered_stock_prices_by_date[(start_date, end_date)] = new_df

In [ ]:
# Sort the dictionary by date ranges
sorted_filtered_prices = sorted(
    filtered_stock_prices_by_date.items(),
    key=lambda x: x[0]
)

In [ ]:
# Create output directory if it does not exist
output_directory = 'Data/FilteredData'
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

In [ ]:
# Save each filtered dataset to a separate Excel file
for i, ((start_date, end_date), df) in enumerate(sorted_filtered_prices, start=1):
    file_name = f"{output_directory}/stock_price_{start_date.date()}_{end_date.date()}.xlsx"
    
    print(f"Saving {file_name}")
    
    # Sort data by date
    df = df.sort_values('Date')
    
    # Save to Excel
    df.to_excel(file_name, index=False)